In [6]:
from langchain_core.documents import Document

In [7]:
# Text data 
# from langchain_community.document_loaders.text import TextLoader

# loader = TextLoader("data/python.txt",encoding="utf-8")

In [8]:
# document = loader.load()

In [9]:
#Pdf data 

# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

### Ingestion Pipeline

In [10]:
# data => documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

def load_all_pdf():
    folder_path = "data/pdf"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path,filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            all_docs.extend(doc)
            num_docs+=1
    print("Total PDF",num_docs)
    print("Total Pages",len(all_docs))
    return all_docs

In [12]:
all_pdf_documents = load_all_pdf()

Total PDF 2
Total Pages 32


In [13]:
#Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents,chunk_size=500,chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_doc = text_splitter.split_documents(documents)
    return chunked_doc




In [14]:
chunks = split_doc(all_pdf_documents)
len(chunks)

321

### Embedding

In [16]:
from sentence_transformers import SentenceTransformer

In [17]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("loading Model ...",self.model_name)
        self.model = SentenceTransformer(self.model_name)

    def generate_embeddings(self,text):
        embeddings = self.model.encode(text,show_progress_bar=True)
        print("embeddings Shape:",embeddings.shape)
        return embeddings


In [20]:
embedding_manager = EmbeddingManager()

loading Model ... all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\Vs code\AIML\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abhay Kumar\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Vector Store

In [18]:
import chromadb
import uuid

In [ ]:
class VectorStoreManager:
    def __init__(self,persist_directory="Data/vector_store",collection_name = "all_pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None 

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory,exist_ok = True)
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        #create the collections
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata={"description":"vector store collection for pdf embedding in RAG"}
        )

        print("initialized the vector store with collections = ",self.collection_name)
        print("document in collections",self.collection.count())
    
    def add_documents(self,documents,embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of document doesnot match num of embeddings")
        
        #store => ids , embedding , document , metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i ,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)
            
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("Total documents added in vector store=", len(documents_content))
        print("Docs in collection:", self.collection.count())

In [25]:
vector_store = VectorStoreManager()

initialized the vector store with collections =  all_pdf_documents
document in collections 0


In [ ]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]
embedding = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings Shape: (321, 384)
total documents added in vector store= 321
docs in collection: 642
